# Week 17: Production RAG & Vector Search
## Notebook 07: RAPTOR: Recursive Abstractive Tree-Organized Retrieval

### 1. Conceptual Intuition & Theoretical Background
Traditional flat chunking (e.g. 500-token chunks) fails when answering **holistic thematic queries** (e.g., "What are the core arguments across this 300-page book?"). Answering high-level queries requires aggregating information spread across hundreds of separate chunks.

**RAPTOR (Recursive Abstractive Processing for Tree-Organized Retrieval)** builds a multi-level hierarchical tree:
1. **Leaf Chunks ($L_0$):** Raw text segmented into standard chunks.
2. **Clustering:** Chunks are clustered using Gaussian Mixture Models (GMM) with soft clustering (allowing chunks to belong to multiple topic clusters).
3. **Abstractive Summarization ($L_1 \dots L_k$):** An LLM summarizes each cluster into an abstractive synopsis node.
4. **Recursive Hierarchy:** The process is repeated recursively until a single root summary is produced.

At query time, retrieval searches **across all tree levels simultaneously** or traverses down relevant subtrees, answering both granular factual questions and broad thematic inquiries.

---

### 2. Scratch Python Implementation: RAPTOR Hierarchical Clustering Tree

In [ ]:
import numpy as np

class SimplifiedRAPTORTree:
    def __init__(self, branch_factor=2):
        self.branch_factor = branch_factor
        self.tree_levels = {}
        
    def build_tree(self, leaf_texts, leaf_embeddings):
        # Level 0: Raw Leaf Chunks
        self.tree_levels[0] = {
            'texts': leaf_texts,
            'embeddings': leaf_embeddings / np.linalg.norm(leaf_embeddings, axis=1, keepdims=True)
        }
        
        current_texts = leaf_texts
        current_embeds = self.tree_levels[0]['embeddings']
        level = 1
        
        while len(current_texts) > 1:
            n_chunks = len(current_texts)
            # Simple pairwise clustering
            n_clusters = max(1, n_chunks // self.branch_factor)
            
            # Group consecutive chunks into summary nodes
            summaries = []
            summary_embeds = []
            
            for c in range(n_clusters):
                start = c * self.branch_factor
                end = min(n_chunks, (c + 1) * self.branch_factor)
                cluster_chunks = current_texts[start:end]
                
                # Mock abstractive summary
                summary_text = f"[Level {level} Summary of {len(cluster_chunks)} nodes: {' '.join(cluster_chunks[:20])}...]"
                summaries.append(summary_text)
                
                # Mean centroid embedding
                mean_emb = np.mean(current_embeds[start:end], axis=0)
                mean_emb /= np.linalg.norm(mean_emb)
                summary_embeds.append(mean_emb)
                
            self.tree_levels[level] = {
                'texts': summaries,
                'embeddings': np.array(summary_embeds)
            }
            
            current_texts = summaries
            current_embeds = np.array(summary_embeds)
            level += 1
            
        print(f"RAPTOR Hierarchical Tree constructed with {len(self.tree_levels)} levels!")
        
    def query(self, query_emb, top_k=2):
        query_emb = query_emb / np.linalg.norm(query_emb)
        retrieved_nodes = []
        
        # Search across all tree levels simultaneously
        for lvl, data in self.tree_levels.items():
            scores = np.dot(data['embeddings'], query_emb)
            best_idx = np.argsort(-scores)[:top_k]
            for idx in best_idx:
                retrieved_nodes.append((lvl, data['texts'][idx], scores[idx]))
                
        # Sort retrieved nodes by cosine similarity
        retrieved_nodes.sort(key=lambda x: x[2], reverse=True)
        return retrieved_nodes[:top_k]

# Generate synthetic document chunks
raw_doc_chunks = [
    "Chapter 1: The foundation of distributed consensus algorithms.",
    "Chapter 2: Paxos mechanics, ballot numbers and leader election.",
    "Chapter 3: Raft protocol simplification, log replication and heartbeat terms.",
    "Chapter 4: Practical Byzantine Fault Tolerance (PBFT) in state machine replication."
]
mock_embeddings = np.random.randn(len(raw_doc_chunks), 16)

raptor = SimplifiedRAPTORTree(branch_factor=2)
raptor.build_tree(raw_doc_chunks, mock_embeddings)

# Query thematic summary
query_vector = np.random.randn(16)
results = raptor.query(query_vector, top_k=3)

print("\n--- RAPTOR Multi-Level Retrieval ---")
for lvl, text, score in results:
    print(f"[Level {lvl}] Score: {score:.4f} -> {text}")

### 3. Step-by-Step Code Walkthrough
1. **Hierarchical Summarization**: Bottom-up aggregation pairs child nodes into higher-level parent summaries.
2. **Multi-Scale Indexing**: Embeds leaf nodes and summary abstractions into a unified retrieval index.
3. **Adaptive Context**: Answers fine-grained questions from leaves and broad questions from tree roots.

---

### 4. Industry Applications & Trade-offs
- **Legal Case Analysis & Research Papers:** Synthesizing arguments across long multi-volume documents.
- **Trade-off:** High indexing compute costs due to recursive LLM summary generation.

---

### 5. Top Interview Questions & Answers
**Q1: Why does standard flat RAG fail on thematic questions like "What are the major themes of this book?"**
> **Answer:** Flat chunking retrieves individual snippets that contain local keywords, missing high-level thematic relationships distributed across the entire text corpus.

**Q2: How does RAPTOR resolve the trade-off between granular retrieval and holistic understanding?**
> **Answer:** It indexes both raw leaf chunks and recursive abstractive summary nodes in the same vector space, allowing the retriever to simultaneously select fine details and high-level syntheses.